# 05. 혼합형(confidence 라우팅 + 교차검증) OCR 검증 파이프라인

## 목적
사용자와의 논의에서 나온 설계를 실제로 구현·실행하여 검증한다.

- **고위험 필드**(요율/한도/수수료/날짜/연락처/시간대 등): confidence와 무관하게 GPT Vision과 Upstage 두 엔진이 **일치해야만 자동 통과**, 불일치 시 사람 검토 큐로.
- **저위험 필드**(정책성 설명 문구): 1개 엔진(GPT Vision)의 **self-reported confidence**가 임계값 이상이면 자동 통과(비용 절감), 미만이면 2번째 엔진과 교차검증.

## 설계 조정 사항 (중요)
1. **Upstage Document Parse 응답에는 필드별 confidence score가 없다.** `notebooks/data/03_ocr_engine_comparison/upstage_raw/*.json`을 직접 확인한 결과, `elements`에는 `id/page/category/content/coordinates`만 있고 `ocr` 키는 단순 불리언 플래그였다. 따라서 "Upstage가 confidence를 준다"는 이전 대화의 일반론은 이 프로젝트의 실제 API 응답 기준으로는 틀렸다. 이 노트북은 Upstage를 confidence 없는 엔진으로 취급하고, Upstage의 원문(markdown)을 LLM으로 구조화하는 **어댑터 방식**으로 필드값을 뽑는다.
2. **GPT Vision(LLM)도 네이티브 confidence가 없다.** 모델에게 0~1 확신도를 말로 보고(verbalized confidence)하게 했다. 이는 연구적으로 실제 정답률과 상관관계가 약하다고 알려진 **약한 신호**이며, 저위험 필드의 비용 절감용 1차 필터로만 쓰고, 고위험 필드의 최종 판단 근거로는 쓰지 않는다.
3. **정답셋 재검증**: 기존 `03_manual_field_gold.json`은 다른 AI 세션이 만든 것이라 신뢰도 우려가 있었다. Claude가 `03/rendered_pages/*.png` 10장을 직접 읽고 43개 필드를 재검증했다 (`05/manual_field_gold_claude_verified.json`). 42/43은 원문과 정확히 일치했고, woori의 `minimum_transaction`은 캡처된 페이지 내에서 명시적 근거를 찾지 못해 낮은 신뢰도로 표시했다.

## 범위
- 다른 노트북/파일은 읽기만 하고 수정하지 않는다. 산출물은 전부 `notebooks/data/05_hybrid_verification_pipeline/` 아래에 새로 만든다.
- 표본은 기존 03 노트북과 동일한 10개 카드사 페이지(43개 필드)를 재사용한다.
- Vision 엔진은 GPT(`gpt-5.4-mini-2026-03-17`)를 사용한다. Claude Vision은 사용하지 않는다 (사용자 지정).


## v2 변경 사항 (실행 후 대화에서 나온 결정)
v1 실행 결과, 저위험 필드는 GPT Vision의 self-reported confidence만으로 Upstage 대조 없이 자동통과시켰는데, `lotte/excluded_loan` 필드에서 GPT가 confidence 0.95로 확신하며 오답("장기카드대출(카드론)")을 낸 것이 그대로 통과되는 사례가 실제로 발생했다.

이에 따라 **v2는 위험도·confidence와 무관하게 모든 필드에서 GPT Vision과 Upstage가 일치해야만 자동통과**하도록 라우팅 로직을 단순화했다 (Option C). confidence는 더 이상 통과 여부를 결정하지 않고, 진단용 참고 데이터로만 남긴다.

이렇게 바꾼 이유(비용 대비 효과):
- 이 표본에서 고위험 필드가 43개 중 37개(86%)라서, "confidence 높으면 Upstage 생략" 방식으로 아낄 수 있는 비용 자체가 애초에 작다.
- 반면 저위험 필드에서 confidence 지름길이 실패한 사례가 이미 1건 확인됐다.
- 비용 절감 효과는 작고 안전성 손실은 확인됐으므로, 이 도메인(금액 정보가 포함된 카드 약관)에서는 지름길을 없애는 쪽이 낫다고 판단했다.

v1 정책이었다면 어떤 결정이 나왔을지는 `decision_v1_confidence_shortcut` 컬럼에 비교용으로 남겨뒀다 (실제 라우팅에는 영향 없음).

**주의**: 이 셀 이하 코드는 구조만 수정했고 아직 재실행하지 않았다. 즉 저장된 출력(있다면)은 v1 기준일 수 있으니, 재실행 전까지는 신뢰하지 않는다.


In [1]:
from pathlib import Path
import base64
import json
import os
import re
import time

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data').is_dir() and (cwd / 'notebooks').is_dir():
    BACKEND_ROOT = cwd
elif (cwd.parent / 'data').is_dir() and cwd.name == 'notebooks':
    BACKEND_ROOT = cwd.parent
else:
    raise RuntimeError(f'프로젝트 루트 또는 notebooks 폴더를 찾을 수 없습니다: {cwd}')

RENDERED_DIR = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison' / 'rendered_pages'
UPSTAGE_RAW_DIR = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison' / 'upstage_raw'
GOLD_PATH = BACKEND_ROOT / 'notebooks' / 'data' / '05_hybrid_verification_pipeline' / 'manual_field_gold_claude_verified.json'
OUTPUT_ROOT = BACKEND_ROOT / 'notebooks' / 'data' / '05_hybrid_verification_pipeline'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from dotenv import load_dotenv
    load_dotenv(BACKEND_ROOT / '.env')
except ImportError:
    pass

from openai import OpenAI

MODEL = "gpt-5.4-mini-2026-03-17"
client = OpenAI()

gold = json.loads(GOLD_PATH.read_text(encoding='utf-8'))
CASES = gold['cases']
assert len(CASES) == 10
total_fields = sum(len(c['fields']) for c in CASES)
print(f'표본: {len(CASES)}페이지, {total_fields}개 필드')
print('경고 API 키 없음' if not os.getenv('OPENAI_API_KEY') else 'OpenAI API 키 확인됨')


표본: 10페이지, 43개 필드
OpenAI API 키 확인됨


In [2]:
# 위험도 계층화 (규칙기반, confidence가 아니라 필드명 패턴 매칭)
HIGH_RISK_KEYWORDS = ['rate', 'limit', 'fee', 'spend', 'cashback', 'transaction', 'date', 'center', 'time']

def is_high_risk(field_name: str) -> bool:
    lname = field_name.lower()
    return any(kw in lname for kw in HIGH_RISK_KEYWORDS)

risk_rows = []
for case in CASES:
    for fname in case['fields']:
        risk_rows.append({'issuer': case['issuer'], 'field': fname, 'risk': 'HIGH' if is_high_risk(fname) else 'LOW'})

risk_df = pd.DataFrame(risk_rows)
print(risk_df['risk'].value_counts())
risk_df.head(10)


risk
HIGH    37
LOW      6
Name: count, dtype: int64


,issuer,field,risk
0,BC,daily_discount_rate,HIGH
1,BC,minimum_spend_1,HIGH
2,BC,integrated_limit_1,HIGH
3,BC,minimum_spend_3,HIGH
4,BC,integrated_limit_3,HIGH
5,NH,benefit_maintenance,LOW
6,NH,launch_date,HIGH
7,NH,customer_center,HIGH
8,hana,department_discount_rate,HIGH
9,hana,department_limit,HIGH


In [3]:
# Engine A: GPT Vision (이미지 직접 읽고 필드값 + self-reported confidence 추출)

def image_to_data_url(image_path: Path) -> str:
    encoded = base64.b64encode(image_path.read_bytes()).decode('utf-8')
    return f'data:image/png;base64,{encoded}'

def find_rendered_image(issuer: str, file_name: str, page_number: int) -> Path:
    stem = file_name.rsplit('.', 1)[0]
    candidate = RENDERED_DIR / f"{issuer}__{stem}__p{page_number:03d}.png"
    if not candidate.exists():
        raise FileNotFoundError(candidate)
    return candidate

def extract_fields_gpt_vision(image_path: Path, field_names: list[str]) -> dict:
    field_list_str = ', '.join(field_names)
    prompt_lines = [
        '이 이미지는 신용카드 상품설명서 PDF의 한 페이지입니다.',
        '아래 필드 각각에 대해, 이미지 원문에서 그 값을 찾아 그대로 옮기세요.',
        '',
        f'필드 목록: {field_list_str}',
        '',
        '규칙:',
        '1. 해석하거나 요약하지 말고, 원문에 보이는 값 그대로 옮기세요 (표기, 단위 포함).',
        '2. 이미지에서 해당 필드값을 찾을 수 없으면 value를 빈 문자열로 두세요.',
        '3. confidence는 당신이 이 값이 맞다고 스스로 확신하는 정도를 0.0~1.0 사이 숫자로 주관적으로 평가하세요.',
        '4. 반드시 아래 JSON 형식으로만 출력하고 다른 설명은 쓰지 마세요.',
        '',
        '{"필드명": {"value": "원문에서 찾은 값", "confidence": 0.0}}',
    ]
    prompt = '\n'.join(prompt_lines)
    response = client.responses.create(
        model=MODEL,
        input=[
            {
                'role': 'user',
                'content': [
                    {'type': 'input_text', 'text': prompt},
                    {'type': 'input_image', 'image_url': image_to_data_url(image_path), 'detail': 'high'},
                ],
            }
        ],
    )
    raw = response.output_text.strip()
    raw = re.sub(r'^```(json)?', '', raw.strip())
    raw = re.sub(r'```$', '', raw.strip()).strip()
    return json.loads(raw)


In [4]:
# Engine B: Upstage 어댑터 (Upstage 원문 markdown -> GPT 텍스트 구조화, confidence 없음)

def load_upstage_text(issuer: str, file_name: str, page_number: int) -> str:
    stem = file_name.rsplit('.', 1)[0]
    candidate = UPSTAGE_RAW_DIR / f"{issuer}__{stem}__p{page_number:03d}.json"
    if not candidate.exists():
        raise FileNotFoundError(candidate)
    raw = json.loads(candidate.read_text(encoding='utf-8'))
    content = raw.get('content', {})
    text = content.get('markdown') or content.get('text') or content.get('html') or ''
    if not text:
        elements_text = []
        for el in raw.get('elements', []):
            ec = el.get('content', {})
            t = ec.get('markdown') or ec.get('text') or ec.get('html') or ''
            if t:
                elements_text.append(t)
        text = '\n'.join(elements_text)
    return text

def extract_fields_upstage_adapter(page_text: str, field_names: list[str]) -> dict:
    field_list_str = ', '.join(field_names)
    prompt_lines = [
        '다음은 Upstage Document Parse가 신용카드 상품설명서 PDF 한 페이지에서 추출한 원문(markdown 또는 html)입니다.',
        '아래 필드 각각에 대해, 이 원문 안에서 그 값을 찾아 그대로 옮기세요.',
        '',
        f'필드 목록: {field_list_str}',
        '',
        '규칙:',
        '1. 해석하거나 요약하지 말고, 원문에 있는 값 그대로 옮기세요.',
        '2. 원문에서 해당 필드값을 찾을 수 없으면 빈 문자열로 두세요.',
        '3. 반드시 아래 JSON 형식으로만 출력하세요 (confidence 없음, 값만).',
        '',
        '{"필드명": "원문에서 찾은 값"}',
        '',
        '--- 원문 시작 ---',
        page_text,
        '--- 원문 끝 ---',
    ]
    prompt = '\n'.join(prompt_lines)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        response_format={'type': 'json_object'},
    )
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)


In [5]:
# 두 엔진을 10페이지 전체에 대해 실행 (문제 발생 시 즉시 예외를 던져 중단)

engine_results = {}  # issuer -> {'vision': {...}, 'upstage': {...}}
errors = []

for case in CASES:
    issuer = case['issuer']
    field_names = list(case['fields'].keys())
    print(f'[처리중] {issuer} ({len(field_names)}개 필드)')

    image_path = find_rendered_image(issuer, case['file_name'], case['page_number'])
    vision_result = extract_fields_gpt_vision(image_path, field_names)

    page_text = load_upstage_text(issuer, case['file_name'], case['page_number'])
    if not page_text.strip():
        raise RuntimeError(f'{issuer}: Upstage 원문이 비어 있음 (raw json 구조 확인 필요)')
    upstage_result = extract_fields_upstage_adapter(page_text, field_names)

    engine_results[issuer] = {'vision': vision_result, 'upstage': upstage_result}
    time.sleep(0.5)

print('=' * 60)
print(f'완료: {len(engine_results)}/{len(CASES)}페이지')


[처리중] BC (5개 필드)


[처리중] NH (3개 필드)


[처리중] hana (5개 필드)


[처리중] hyundai (4개 필드)


[처리중] ibk (5개 필드)


...

[처리중] lotte (3개 필드)


[처리중] samsung (5개 필드)


[처리중] shinhan (6개 필드)


[처리중] woori (4개 필드)


완료: 10/10페이지


In [6]:
# 값 정규화 및 일치 판정

def normalize(s) -> str:
    return re.sub(r'[\s,]', '', str(s or ''))

def values_match(a, b) -> bool:
    na, nb = normalize(a), normalize(b)
    if not na or not nb:
        return False
    return na == nb or na in nb or nb in na

CONFIDENCE_THRESHOLD = 0.8


In [7]:
# 라우팅 로직 v2: confidence 지름길 제거, 위험도/confidence 무관하게 항상 교차검증 (Option C)
# v1(이전 버전)은 저위험+고confidence 필드를 Upstage 대조 없이 자동통과시켰으나,
# lotte/excluded_loan 사례에서 GPT가 confidence 0.95로 확신한 오답이 그대로 통과되는 것을 확인했다.
# v2는 confidence를 진단용 참고 데이터로만 기록하고, 통과 여부 결정에는 쓰지 않는다.

routing_rows = []

for case in CASES:
    issuer = case['issuer']
    vision = engine_results[issuer]['vision']
    upstage = engine_results[issuer]['upstage']

    for fname, gold_value in case['fields'].items():
        risk = 'HIGH' if is_high_risk(fname) else 'LOW'
        v_entry = vision.get(fname, {}) if isinstance(vision.get(fname), dict) else {}
        v_value = v_entry.get('value', '')
        v_conf = v_entry.get('confidence', None)
        u_value = upstage.get(fname, '')

        agree = values_match(v_value, u_value)

        # v2 결정: 위험도/confidence와 무관하게 항상 교차검증 결과로만 결정
        if agree:
            decision = 'AUTO_PASS'
            final_value = v_value
        else:
            decision = 'HUMAN_REVIEW'
            final_value = None

        # v1(이전 정책, confidence 지름길 있었던 버전) 참고용 재계산 - 실제 라우팅에는 미반영
        if risk == 'HIGH':
            decision_v1 = 'AUTO_PASS' if agree else 'HUMAN_REVIEW'
        else:
            if v_conf is not None and v_conf >= CONFIDENCE_THRESHOLD:
                decision_v1 = 'AUTO_PASS_LOW_RISK_SINGLE_ENGINE'
            elif agree:
                decision_v1 = 'AUTO_PASS'
            else:
                decision_v1 = 'HUMAN_REVIEW'

        correct_vs_gold = values_match(final_value, gold_value) if final_value is not None else None
        vision_correct_vs_gold = values_match(v_value, gold_value)

        routing_rows.append({
            'issuer': issuer, 'field': fname, 'risk': risk,
            'gold_value': gold_value, 'vision_value': v_value, 'vision_confidence': v_conf,
            'upstage_value': u_value, 'engines_agree': agree,
            'decision': decision, 'final_value': final_value,
            'auto_pass_correct': correct_vs_gold,
            'vision_confident_but_wrong': bool(v_conf is not None and v_conf >= CONFIDENCE_THRESHOLD and not vision_correct_vs_gold),
            'decision_v1_confidence_shortcut': decision_v1,
            'policy_changed_vs_v1': decision != decision_v1,
        })

routing_df = pd.DataFrame(routing_rows)
routing_df


,issuer,field,risk,gold_value,vision_value,vision_confidence,upstage_value,engines_agree,decision,final_value,auto_pass_correct,vision_confident_but_wrong,decision_v1_confidence_shortcut,policy_changed_vs_v1
0,BC,daily_discount_rate,HIGH,7%,매일 할인 7%,0.99,매일 할인 7%,True,AUTO_PASS,매일 할인 7%,True,False,AUTO_PASS,False
1,BC,minimum_spend_1,HIGH,15만원 이상,1만원 이상,0.98,15만원 이상,False,HUMAN_REVIEW,NaN,None,True,HUMAN_REVIEW,False
2,BC,integrated_limit_1,HIGH,"5,000원",,0.02,"5,000원",False,HUMAN_REVIEW,NaN,None,False,HUMAN_REVIEW,False
3,BC,minimum_spend_3,HIGH,50만원 이상,15만원 이상,0.97,50만원 이상,False,HUMAN_REVIEW,NaN,None,True,HUMAN_REVIEW,False
4,BC,integrated_limit_3,HIGH,"20,000원","5,000원",0.95,"20,000원",False,HUMAN_REVIEW,NaN,None,True,HUMAN_REVIEW,False
5,NH,benefit_maintenance,LOW,3년 이상,,0.18,,False,HUMAN_REVIEW,NaN,None,False,HUMAN_REVIEW,False
6,NH,launch_date,HIGH,2026년 1월 28일,,0.16,2026년 1월 28일,False,HUMAN_REVIEW,NaN,None,False,HUMAN_REVIEW,False
7,NH,customer_center,HIGH,1644-4000,카드고객상담센터 1644-4000,0.97,카드고객상담센터 1644-4000 해외에서 문의 시 82-2-6942-6478,True,AUTO_PASS,카드고객상담센터 1644-4000,True,False,AUTO_PASS,False
8,hana,department_discount_rate,HIGH,5%,,0.14,5%,False,HUMAN_REVIEW,NaN,None,False,HUMAN_REVIEW,False
9,hana,department_limit,HIGH,"20,000원 / 월",,0.13,"20,000원 / 월",False,HUMAN_REVIEW,NaN,None,False,HUMAN_REVIEW,False


In [8]:
# 파이프라인 성능 지표 (v2 기준) + v1(confidence 지름길) 대비 비교

n_total = len(routing_df)
auto_pass_mask = routing_df['decision'].str.startswith('AUTO_PASS')
n_auto = int(auto_pass_mask.sum())
n_human = int((~auto_pass_mask).sum())

stp_rate = n_auto / n_total * 100
auto_pass_accuracy = routing_df.loc[auto_pass_mask, 'auto_pass_correct'].mean() * 100 if n_auto else float('nan')
human_queue_rate = n_human / n_total * 100
confident_wrong_count = int(routing_df['vision_confident_but_wrong'].sum())

# v1 정책 기준 재계산 (비교용)
v1_auto_mask = routing_df['decision_v1_confidence_shortcut'].str.startswith('AUTO_PASS')
n_auto_v1 = int(v1_auto_mask.sum())
v1_correct = routing_df.loc[v1_auto_mask].apply(
    lambda r: values_match(r['vision_value'] if r['decision_v1_confidence_shortcut'] == 'AUTO_PASS_LOW_RISK_SINGLE_ENGINE' else r['final_value'], r['gold_value'])
    if r['decision_v1_confidence_shortcut'] == 'AUTO_PASS_LOW_RISK_SINGLE_ENGINE'
    else r['auto_pass_correct'],
    axis=1,
)
stp_rate_v1 = n_auto_v1 / n_total * 100
auto_pass_accuracy_v1 = v1_correct.mean() * 100 if n_auto_v1 else float('nan')
policy_changed_count = int(routing_df['policy_changed_vs_v1'].sum())

summary = {
    'total_fields': n_total,
    'stp_rate_pct': round(stp_rate, 2),
    'auto_pass_accuracy_pct': round(auto_pass_accuracy, 2) if n_auto else None,
    'human_queue_rate_pct': round(human_queue_rate, 2),
    'human_queue_count': n_human,
    'vision_confident_but_wrong_count': confident_wrong_count,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'policy': 'v2_always_cross_validate',
    'v1_comparison': {
        'stp_rate_pct': round(stp_rate_v1, 2),
        'auto_pass_accuracy_pct': round(auto_pass_accuracy_v1, 2) if n_auto_v1 else None,
        'fields_where_policy_changed': policy_changed_count,
    },
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

print()
print('=== 위험도별 결정 분포 (v2) ===')
print(routing_df.groupby(['risk', 'decision']).size())

print()
print('=== v1 -> v2 정책 변경으로 결과가 달라진 필드 ===')
changed = routing_df.loc[routing_df['policy_changed_vs_v1'], ['issuer', 'field', 'risk', 'decision_v1_confidence_shortcut', 'decision', 'gold_value', 'vision_value']]
print(changed.to_string(index=False) if len(changed) else '(없음)')

print()
print('=== 사람 검토 큐로 간 필드 (실제 운영이라면 여기까지만 사람이 봄) ===')
print(routing_df.loc[~auto_pass_mask, ['issuer', 'field', 'risk', 'gold_value', 'vision_value', 'upstage_value']].to_string(index=False))


{
  "total_fields": 43,
  "stp_rate_pct": 60.47,
  "auto_pass_accuracy_pct": 92.31,
  "human_queue_rate_pct": 39.53,
  "human_queue_count": 17,
  "vision_confident_but_wrong_count": 8,
  "confidence_threshold": 0.8,
  "policy": "v2_always_cross_validate",
  "v1_comparison": {
    "stp_rate_pct": 60.47,
    "auto_pass_accuracy_pct": 92.31,
    "fields_where_policy_changed": 2
  }
}

=== 위험도별 결정 분포 (v2) ===
risk  decision    
HIGH  AUTO_PASS       24
      HUMAN_REVIEW    13
LOW   AUTO_PASS        2
      HUMAN_REVIEW     4
dtype: int64

=== v1 -> v2 정책 변경으로 결과가 달라진 필드 ===
 issuer         field risk  decision_v1_confidence_shortcut  decision    gold_value                 vision_value
kookmin   family_card  LOW AUTO_PASS_LOW_RISK_SINGLE_ENGINE AUTO_PASS    가족카드 발급 불가 KB국민 My WE:SH 카드는 가족카드 발급 불가
  lotte excluded_loan  LOW AUTO_PASS_LOW_RISK_SINGLE_ENGINE AUTO_PASS 단기카드대출(현금서비스)   단기카드대출(현금서비스), 장기카드대출(카드론)

=== 사람 검토 큐로 간 필드 (실제 운영이라면 여기까지만 사람이 봄) ===
 issuer                    field risk

In [9]:
# 결과 저장

routing_df.to_json(OUTPUT_ROOT / 'routing_results.json', orient='records', force_ascii=False, indent=2)
(OUTPUT_ROOT / 'pipeline_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('저장 완료:')
print(' -', OUTPUT_ROOT / 'routing_results.json')
print(' -', OUTPUT_ROOT / 'pipeline_summary.json')


저장 완료:
 - notebooks/data/05_hybrid_verification_pipeline/routing_results.json
 - notebooks/data/05_hybrid_verification_pipeline/pipeline_summary.json
